In [8]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

housing = fetch_openml('california_housing', version=1, as_frame=True)
X = housing.data
y=housing.target.astype(float)

X = pd.get_dummies(X, drop_first=True)

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def print_rf_performance(n_estimators):
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
    rf.fit(X_train, y_train)


    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))


    cv_mse = cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_mse).mean()

    print(f"\nRandomForestRegressor (n_estimators={n_estimators})")
    print(f"Train RMSE:       {train_rmse:.3f}")
    print(f"Test RMSE:        {test_rmse:.3f}")
    print(f"5-Fold CV RMSE:   {cv_rmse:.3f}")


print_rf_performance(n_estimators=100)


print_rf_performance(n_estimators=500)


RandomForestRegressor (n_estimators=100)
Train RMSE:       18091.111
Test RMSE:        48988.274
5-Fold CV RMSE:   49324.577


KeyboardInterrupt: 

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd


housing = fetch_openml('california_housing', version=1, as_frame=True)
X = housing.data
y = housing.target.astype(float)
X = pd.get_dummies(X, drop_first=True)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)


best_params = grid_search.best_params_
best_cv_mse = -grid_search.best_score_
best_cv_rmse = np.sqrt(best_cv_mse)
print("Best Parameters:", best_params)
print(f"Best 5-Fold CV RMSE: {best_cv_rmse:.4f}")


best_model = grid_search.best_estimator_
test_pred = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print(f"Test RMSE (best model): {test_rmse:.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best 5-Fold CV RMSE: 49217.5891
Test RMSE (best model): 48833.1619


In [1]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd


housing = fetch_openml('california_housing', version=1, as_frame=True)
df = housing.frame.copy()  # Use the full DataFrame


X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]


X["Rooms_per_household"] = X["total_rooms"] / X["households"]
X["Bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]


X = pd.get_dummies(X, drop_first=True)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

KeyError: "['MedHouseVal'] not found in axis"

In [7]:
X_base = df.drop(columns=["MedHouseVal", "total_rooms", "total_bedrooms", "households", "ocean_proximity"]).copy()
# Add back necessary features
X_base["total_rooms"] = df["total_rooms"]
X_base["total_bedrooms"] = df["total_bedrooms"]
X_base["households"] = df["households"]
X_base = pd.get_dummies(X_base, drop_first=True)
X_base_imputed = imputer.fit_transform(X_base)

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base_imputed, y, test_size=0.2, random_state=42
)

rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_train_base, y_train_base)
base_pred = rf_base.predict(X_test_base)
base_rmse = np.sqrt(mean_squared_error(y_test_base, base_pred))

KeyError: "['MedHouseVal'] not found in axis"

In [ ]:
rf_new = RandomForestRegressor(random_state=42)
rf_new.fit(X_train, y_train)
new_pred = rf_new.predict(X_test)
new_rmse = np.sqrt(mean_squared_error(y_test, new_pred))

In [ ]:
print(f"RMSE (Before feature engineering): {base_rmse:.4f}")
print(f"RMSE (After feature engineering): {new_rmse:.4f}")